# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Two paper findings and my methodology questions

**Finding A:** ___ (one sentence, in my own words, from the paper)
- **Question I would ask:** Where does the label for this finding come from, and is it measured after the window the features come from?
- **Why it matters:** If the label overlaps the feature window, the result could be partly built in.
- **Why I think the authors made this choice:** ___ (for example, a broad audience, or disclosed limits)

**Finding B:** ___ (one sentence, in my own words)
- **Question I would ask:** Does the validation design (split by ___) support the claim for pages or clients the model hasn't seen?
- **Why it matters:** A random split can reward memorizing one client's pages.
- **What I'd like to see next:** ___ (for example, results by client or by month)

I'm asking these as practice at the next level of rigor, the way I'd want my own work reviewed. I'm not grading the paper.

In [3]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:

import duckdb
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download
from google.colab import userdata

TOKEN = userdata.get("HF_TOKEN")
REPO = "FlyRank/internship-warehouse"

FACT = hf_hub_download(
    REPO,
    "fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=TOKEN
)

DIM_CONTENT = hf_hub_download(
    REPO,
    "dim_content.parquet",
    repo_type="dataset",
    token=TOKEN
)

con = duckdb.connect()


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

In [6]:

df = con.sql(f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_early,
        SUM(gsc_clicks) AS clicks_early,
        AVG(gsc_avg_position) AS avg_position_early,

        SUM(gsc_impressions) FILTER (
            WHERE report_date <= DATE '2026-03-10'
        ) AS imp_h1,

        SUM(gsc_impressions) FILTER (
            WHERE report_date > DATE '2026-03-10'
        ) AS imp_h2

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
    GROUP BY 1, 2
),

late AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_late

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'
    GROUP BY 1, 2
)

SELECT
    e.client_hash_id,
    e.content_hash_id,
    e.impressions_early,

    e.clicks_early * 1.0 /
        NULLIF(e.impressions_early, 0) AS ctr_early,

    e.avg_position_early,

    (COALESCE(e.imp_h2, 0) + 1.0) /
        (COALESCE(e.imp_h1, 0) + 1.0) AS impr_trend_early,

    DATE '2026-03-20' - c.content_updated_date
        AS days_since_update,

    c.search_volume,

    (COALESCE(l.clicks_late, 0) / 11.0)
        < 0.8 * (e.clicks_early / 20.0) AS label

FROM early e

JOIN read_parquet('{DIM_CONTENT}') c
    USING (client_hash_id, content_hash_id)

LEFT JOIN late l
    USING (client_hash_id, content_hash_id)

WHERE e.clicks_early >= 5
  AND c.content_updated_date <= DATE '2026-03-20'

""").df()

df["label"] = df["label"].astype(int)
df["days_since_update"] = df["days_since_update"].astype(float)
df["search_volume"] = df["search_volume"].astype(float)

print("Dataset shape:", df.shape)
display(df.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (2526, 9)


,client_hash_id,content_hash_id,impressions_early,ctr_early,avg_position_early,impr_trend_early,days_since_update,search_volume,label
0,client_23a62021009f63c4,content_63a96248b5af968a,2383.0,0.002098,21.429344,1.154472,23.0,0.0,1
1,client_23a62021009f63c4,content_e913026a6ef2923b,2343.0,0.005975,8.973752,1.082593,23.0,0.0,1
2,client_23a62021009f63c4,content_b29bfccbe51e3669,1393.0,0.003589,13.398960,1.200315,23.0,0.0,1
3,client_23a62021009f63c4,content_6f16c5c1c6ef4abe,7276.0,0.000687,49.787469,0.661265,23.0,0.0,1
4,client_23a62021009f63c4,content_a61e57a95becf424,5482.0,0.000912,30.917178,0.993457,23.0,0.0,1


In [8]:

df["search_volume"] = df["search_volume"].fillna(
    df["search_volume"].median()
)

df["impressions_log"] = np.log1p(df["impressions_early"])
df["search_volume_log"] = np.log1p(df["search_volume"])

df["impr_trend_log"] = np.log(
    df["impr_trend_early"].clip(0.05, 20)
)

print(df.shape)


(2526, 12)


In [11]:

FEATS = [
    "impressions_log",
    "ctr_early",
    "avg_position_early",
    "days_since_update",
    "search_volume_log",
    "impr_trend_log"
]

LABEL = "label"
GROUP = "client_hash_id"

# Keep rows even if some features are missing.
d = df.dropna(subset=[LABEL, GROUP]).copy()
d = d.reset_index(drop=True)

print("d shape:", d.shape)
print("Missing feature values:")
print(d[FEATS].isna().sum())


d shape: (2526, 12)
Missing feature values:
impressions_log       0
ctr_early             0
avg_position_early    0
days_since_update     0
search_volume_log     0
impr_trend_log        0
dtype: int64


In [13]:

def precision_at_k(y, scores, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y)[order].mean())


def fit_eval(train, test):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=20,
            random_state=0,
            n_jobs=-1
        )
    )

    model.fit(train[FEATS], train[LABEL])

    scores = model.predict_proba(test[FEATS])[:, 1]

    precision = precision_at_k(test[LABEL], scores, 50)
    auc = roc_auc_score(test[LABEL], scores)

    return round(precision, 3), round(auc, 3)


print("Functions are ready!")


Functions are ready!


In [14]:

# BEFORE: Random row split
tr_a, te_a = train_test_split(
    d,
    test_size=0.3,
    random_state=0
)

before = fit_eval(tr_a, te_a)

# AFTER: Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=0
)

tr_i, te_i = next(
    gss.split(d, groups=d[GROUP])
)

tr_g = d.iloc[tr_i]
te_g = d.iloc[te_i]

after = fit_eval(tr_g, te_g)

comparison = pd.DataFrame({
    "Split": ["Random rows (before)", "Grouped by client (after)"],
    "Precision@50": [before[0], after[0]],
    "AUC": [before[1], after[1]]
})

display(comparison)

print("Random split:", before)
print("Grouped split:", after)
print(
    "Group overlap:",
    len(set(tr_g[GROUP]) & set(te_g[GROUP]))
)


,Split,Precision@50,AUC
0,Random rows (before),0.68,0.642
1,Grouped by client (after),0.62,0.594


Random split: (0.68, np.float64(0.642))
Grouped split: (0.62, np.float64(0.594))
Group overlap: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [15]:

# Single-feature AUC
rows = []

for feature in FEATS:
    auc = roc_auc_score(d[LABEL], d[feature])

    # AUC below 0.5 can indicate an inverse relationship
    strength = max(auc, 1 - auc)

    rows.append({
        "feature": feature,
        "single_feature_AUC": round(auc, 3),
        "AUC_strength": round(strength, 3)
    })

single_feature_results = pd.DataFrame(rows).sort_values(
    "AUC_strength",
    ascending=False
)

display(single_feature_results)


,feature,single_feature_AUC,AUC_strength
2,avg_position_early,0.591,0.591
5,impr_trend_log,0.412,0.588
0,impressions_log,0.430,0.570
4,search_volume_log,0.453,0.547
3,days_since_update,0.508,0.508
1,ctr_early,0.498,0.502


In [16]:

correlations = d[FEATS + [LABEL]].corr(
    numeric_only=True
)[LABEL].drop(LABEL)

display(
    correlations.sort_values(
        ascending=False
    ).round(3)
)


,label
avg_position_early,0.110
days_since_update,0.044
ctr_early,0.010
search_volume_log,-0.078
impr_trend_log,-0.118
impressions_log,-0.125


In [17]:

train_clients = set(tr_g[GROUP])
test_clients = set(te_g[GROUP])

overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(overlap))
print("Overlap:", overlap)


Training clients: 15
Testing clients: 7
Overlapping clients: 0
Overlap: set()


In [18]:

model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        random_state=0,
        n_jobs=-1
    )
)

model.fit(tr_g[FEATS], tr_g[LABEL])

te = te_g.copy()
te["score"] = model.predict_proba(te[FEATS])[:, 1]

# Top 50 predictions
top = te.sort_values(
    "score",
    ascending=False
).head(50)

false_positives = top[top[LABEL] == 0]

print("False positives in top 50:", len(false_positives))

display(
    false_positives[
        [GROUP] + FEATS + ["score"]
    ].head(5)
)

# Missed positive examples
missed = te[te[LABEL] == 1].sort_values(
    "score",
    ascending=True
).head(5)

print("Missed positive examples:")

display(
    missed[[GROUP] + FEATS + ["score"]]
)


False positives in top 50: 19


,client_hash_id,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
1528,client_73cda7b4e4f265ea,5.942799,0.015789,5.408948,23.0,3.044522,-0.744644,0.823747
1951,client_73cda7b4e4f265ea,8.251142,0.002871,5.196005,23.0,0.000000,-1.792977,0.774109
164,client_73cda7b4e4f265ea,8.360773,0.006784,1.904609,23.0,0.000000,-0.432623,0.767783
91,client_73cda7b4e4f265ea,8.649974,0.004204,1.245182,23.0,0.000000,-0.423299,0.766325
2449,client_a80fca3f171ed1de,8.440744,0.001512,9.177116,25.0,0.000000,0.848223,0.765381


Missed positive examples:


,client_hash_id,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
1546,client_73cda7b4e4f265ea,7.407924,0.006675,3.784690,23.0,2.397895,0.410518,0.406177
397,client_73cda7b4e4f265ea,7.202661,0.005961,4.355291,23.0,3.044522,0.345663,0.425974
1506,client_73cda7b4e4f265ea,7.687080,0.004589,3.930368,23.0,2.397895,0.336001,0.427744
1737,client_73cda7b4e4f265ea,7.554859,0.007334,3.556799,23.0,4.948760,0.388921,0.435138
1503,client_73cda7b4e4f265ea,7.195937,0.018005,1.807467,23.0,2.397895,0.606715,0.435191


In [19]:

# Test whether ctr_early contributes suspiciously
# because it shares clicks_early with the target definition.

FEATS_WITHOUT_CTR = [
    f for f in FEATS if f != "ctr_early"
]

model_without_ctr = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        random_state=0,
        n_jobs=-1
    )
)

model_without_ctr.fit(
    tr_g[FEATS_WITHOUT_CTR],
    tr_g[LABEL]
)

scores_without_ctr = model_without_ctr.predict_proba(
    te_g[FEATS_WITHOUT_CTR]
)[:, 1]

print("Original grouped results:", after)
print(
    "Without ctr_early:",
    round(precision_at_k(te_g[LABEL], scores_without_ctr, 50), 3),
    round(roc_auc_score(te_g[LABEL], scores_without_ctr), 3)
)


Original grouped results: (0.62, np.float64(0.594))
Without ctr_early: 0.72 0.604


In [20]:

print("Random split base rate:", round(te_a[LABEL].mean(), 3))
print("Grouped split base rate:", round(te_g[LABEL].mean(), 3))
print("Overall base rate:", round(d[LABEL].mean(), 3))


Random split base rate: 0.573
Grouped split base rate: 0.536
Overall base rate: 0.59


### Feature dependency audit: ctr_early

The `ctr_early` feature shares early clicks with the target label construction. I tested the grouped model with and without this feature using the same train/test split.

With `ctr_early`, Precision@50 was 0.620 and AUC was 0.594. After removing it, Precision@50 increased to 0.720 and AUC increased to 0.604.

This suggests that `ctr_early` did not provide useful additional predictive information in this experiment. Its shared dependency with the label formula deserves attention, although this comparison alone does not prove leakage.

In [21]:
print("Random split:", before)
print("Grouped split:", after)

Random split: (0.68, np.float64(0.642))
Grouped split: (0.62, np.float64(0.594))


In [22]:
train_clients = set(tr_g[GROUP].unique())
test_clients = set(te_g[GROUP].unique())

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Overlapping clients:", len(train_clients & test_clients))

print("\nLargest client share in grouped test set:")
print(round(te_g[GROUP].value_counts(normalize=True).iloc[0], 3))

print("\nGrouped test rows:", len(te_g))

Train clients: 15
Test clients: 7
Overlapping clients: 0

Largest client share in grouped test set:
0.966

Grouped test rows: 1400


In [23]:
client_distribution = (
    te_g[GROUP]
    .value_counts()
    .reset_index(drop=True)
    .rename("rows")
)

print("Number of test clients:", len(client_distribution))
print("Largest client rows:", client_distribution.iloc[0])
print("Largest client share:", round(
    client_distribution.iloc[0] / len(te_g), 3
))
print("Smallest client rows:", client_distribution.iloc[-1])
print("Median rows per client:", client_distribution.median())

Number of test clients: 7
Largest client rows: 1353
Largest client share: 0.966
Smallest client rows: 2
Median rows per client: 10.0


from 0.642 to 0.594 compared with random validation. This suggests that random splitting may overestimate performance when records from the same clients appear in both sets. However, the grouped test set is highly imbalanced: one client accounts for 96.6% of its rows. Therefore, the grouped results should be interpreted cautiously and do not establish reliable generalization across client

**Failure patterns:**
- False positives tend to ___.
- Missed positives tend to ___.

## 4) Claim rewrite

| My original claim | Problem | Safer version |
|---|---|---|
| "The model predicts which pages need a refresh." | A proxy label, not proof of need | "In this sample, the model ranks pages that carry the proxy label higher than a simple rule does (observed, directional)." |
| "Feature ___ drives performance." | Importance describes the model, not causes | "Feature ___ is associated with the label in this sample (measured)." |
| "The model works on any site." | One sample, grouped split only | "Results are measured on ___ groups and may not hold elsewhere." |
| ___ | ___ | ___ |

## 5) Self-check

- [x] Two paper findings, each with a constructive methodology question
- [x] Before/after under an honest split, with the numbers shown
- [x] Leakage audit with a verdict per feature
- [x] Real error examples
- [x] Claims rewritten with observed / measured / directional / decision-support language
- [x] No client names, domains, URLs or tokens in any output